# Run experiments
Run traditional and deep learning pipelines on the ABIDE dataset. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Process datasets in `product/scripts/run_scripts.ipynb` before running experiments.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
# Change runtime type to GPU
import sys
if 'google.colab' in sys.modules:
    sys.path.append('/content/drive/MyDrive/PROJECT/product/src')

    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    # Install deps
    !pip install -q numpy pandas torch torch-geometric scikit-learn torchmetrics optuna

    # Add project modules
    sys.path.insert(0, './product/src')

In [ ]:
# Parallel folds. Prevent each worker from spawning its own thread pool.
# Use only when n_jobs > 1.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from evaluation.validation import stratified_k_fold, leave_one_site_out, nested_stratified_k_fold, nested_leave_one_site_out, final_study
from data_io.centralised_logger import CentralisedLogger
from sklearn.decomposition import PCA
from autoencoder.model_transformer import SDAETransformer
import torch
from data_io.save_load_dataset import load_dataset, mixed_subset
from gcn.model_transformer import GCNTransformer
from gcn.build_population_transformer import PopulationGraphTransformer
from evaluation.validation import log_worker
from utils.paths import get_project_root
from pathlib import Path

## 2. Load Dataset **[CONFIGURE]**
Set `prefix` to the folder name the processed dataset is under.
To use the sex-balanced subset instead, set the prefix to `mixed`.

> **NOTE: using the mixed subset requires downloading and processing the female and male age-matched subsets in `run_scripts.ipynb`. Instructions are provided in the notebook.**

In [ ]:
prefix = "NYU_UCLA_PITT"
if prefix == "mixed":
    X, y, metadata, _ = mixed_subset()
else:
    X, _, y, metadata, _ = load_dataset(prefix, verbose=True)

## 3. Device Selection

Auto-selects MPS (Apple Silicon), CUDA (GPU) or CPU depending on hardware availability. Override by uncommenting `device = "cpu"` if needed.

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(f"Using device: {device}")

## 4. Pipeline Configuration **[CONFIGURE]**

Set `technique`, `graph`, and `model` to select the experiment to run.

| Variable | Options |
|---|---|
| `technique` | `'selectkbest'`, `'pca'`, `'sdae'` |
| `graph` | `'population'` (used only when `model == 'gcn'`) |
| `model` | `'svm'`, `'rf'`, `'xgb'`, `'gcn'` |

Deep learning components (`sdae`, `gcn`) force `n_jobs=1` automatically to prevent OOM.

In [ ]:
# ---- CHANGE THESE ------------------------------------------------
technique = 'selectkbest'
graph = 'population'
model = 'svm'
# ------------------------------------------------------------------

component_library = {
    'techniques': {
        'selectkbest': SelectKBest(f_classif, k=1000),
        'pca': PCA(n_components=0.95),
        'sdae': SDAETransformer(device="cpu"),
    },
    'classifiers': {
        'svm': SVC(class_weight='balanced', probability=True, random_state=42),
        'rf': RandomForestClassifier(),
        'xgb': XGBClassifier(),
        'gcn': GCNTransformer(device=device)
    },
    'graphs': {
        'population': PopulationGraphTransformer(phenotypic_scores=['SEX', 'AGE_AT_SCAN', 'FIQ'], threshold_percentile=75.0)
    }
}

reduction_spaces = {
    'selectkbest': {
        'selectkbest__k': ('categorical', [1000, 2000, 3000, 4000, 5000]),
    },
    'pca': {
        'pca__n_components': ('float', 0.85, 0.95)
    },
    'sdae': {
        'sdae__ae_hidden_dim':          ('categorical', [512, 1024]),
        'sdae__ae_latent_dim':          ('categorical', [64, 128, 256]),
        'sdae__ae_input_dropout':       ('float', 0.0, 0.5),
        'sdae__ae_hidden_dropout':      ('float', 0.1, 0.5),
        'sdae__alpha':                  ('float', 0.5, 2.0),
        'sdae__ae1_epochs':             ('categorical', [100]),
        'sdae__ae1_lr':                 ('float', 1e-4, 1e-2),
        'sdae__ae1_weight_decay':       ('float', 1e-5, 1e-3),
        'sdae__ae1_clf_weight_decay':   ('float', 1e-3, 1e-1),
        'sdae__ae2_epochs':             ('categorical', [50]),
        'sdae__ae2_patience':           ('categorical', [30]),
        'sdae__ae2_lr':                 ('float', 1e-5, 1e-3),
        'sdae__ae2_weight_decay':       ('float', 1e-5, 1e-3),
        'sdae__ae2_factor':             ('categorical', [0.5]),
        'sdae__ae2_scheduler_patience': ('categorical', [30]),
    }
}

clf_spaces = {
    'svm': {
        'svm__C': ('float', 1e-5, 100.0, True),
        'svm__kernel': ('categorical', ['linear', 'rbf', 'poly']),
        'svm__gamma': ('float', 1e-6, 1e-1, True),
        'svm__degree': ('int', 2, 4),
        'svm__coef0': ('float', 0.0, 5.0),
    },
    'rf': {
        'rf__n_estimators': ('int', 100, 1000),
        'rf__max_depth': ('int', 3, 30),
        'rf__min_samples_split': ('int', 2, 15),
        'rf__min_samples_leaf': ('int', 1, 7),
        'rf__max_features': ('categorical', ['sqrt']),
        'rf__bootstrap': ('categorical', [True]),
        'rf__criterion': ('categorical', ['gini']),
    },
    'xgb': {
        'xgb__n_estimators': ('int', 100, 1000),
        'xgb__max_depth': ('int', 3, 12),
        'xgb__min_child_weight': ('int', 1, 10),
        'xgb__learning_rate': ('float', 0.01, 0.3, True),
        'xgb__gamma': ('float', 1e-8, 1.0, True),
        'xgb__subsample': ('float', 0.5, 1.0),
        'xgb__colsample_bytree': ('float', 0.5, 1.0),
        'xgb__reg_alpha': ('float', 1e-8, 100.0, True),
        'xgb__reg_lambda': ('float', 1e-8, 100.0, True),
    },
    'gcn': {
        'gcn__hidden_dim': ('categorical', [16, 32, 64]),
        'gcn__dropout': ('float', 0.2, 0.5),
        'gcn__lr': ('float', 1e-4, 1e-2, True),
        'gcn__weight_decay': ('float', 1e-6, 1e-2, True),
        'gcn__epochs': ('categorical', [150]),
        'gcn__factor': ('categorical', [0.5]),
        'gcn__patience': ('categorical', [30]),
    }
}

graph_spaces = {
    'population': {
        'population__threshold_percentile': ('float', 50.0, 95.0)
    }
}

heavy_components = {'gcn', 'sdae'}

if technique in heavy_components or model in heavy_components:
    n_jobs = 1
    log_worker("Deep Learning component detected. Setting n_jobs to 1.")
else:
    n_jobs = -1
    log_worker("Standard components detected. Setting n_jobs to -1.")

# Add graph to pipeline if selected model is ResGCN
if model == 'gcn':
    pipeline = Pipeline([
        (technique, component_library['techniques'][technique]),
        (graph, component_library['graphs'][graph]),
        (model, component_library['classifiers'][model])
    ])
    pipeline.named_steps['population'].metadata = metadata
    param_grid = {**reduction_spaces[technique], **graph_spaces[graph], **clf_spaces[model]}
else:
    pipeline = Pipeline([
        (technique, component_library['techniques'][technique]),
        (model, component_library['classifiers'][model])
    ])
    param_grid = {**reduction_spaces[technique], **clf_spaces[model]}

logger = CentralisedLogger(model_name=f'{model}_{technique}')

root = get_project_root()
db_path = Path(f"{root}/product/results/artefacts/{model}_{technique}/{prefix}_optuna.db")
db_path.parent.mkdir(parents=True, exist_ok=True)
storage = f"sqlite:///{db_path.absolute()}"

## 5. Baseline cross validation

Runs Stratified K-Fold (5 folds) and Leave-One-Site-Out using default hyperparameters. Run these first to get a baseline before optimisation.

In [ ]:
stratified_k_fold(X=X, y=y, metadata=metadata, model=pipeline, logger=logger, n_folds=5, device=device, n_jobs=n_jobs)

In [ ]:
leave_one_site_out(X=X, y=y, metadata=metadata, model=pipeline, logger=logger, device=device, n_jobs=n_jobs)

## 6. Nested cross validation

Runs Optuna hyperparameter search inside each fold. Results are saved to the SQLite database defined in Section 4. Expect long runtimes for SDAE and GCN pipelines.

In [ ]:
nested_stratified_k_fold(X=X, y=y, metadata=metadata, model=pipeline, param_grid=param_grid, n_trials=50, logger=logger, device=device, n_jobs=n_jobs, storage=storage)

In [ ]:
nested_leave_one_site_out(X=X, y=y, metadata=metadata, model=pipeline, param_grid=param_grid, n_trials=50, logger=logger, device=device, n_jobs=n_jobs, storage=storage)

## 7. Final model

Trains on full dataset. Produces deployable model artefact configurations.

In [ ]:
final_study(X=X, y=y, metadata=metadata, model=pipeline, param_grid=param_grid, n_trials=100, logger=logger, device=device, n_jobs=n_jobs)